In [9]:
import pandas as pd
import numpy as np
from os.path import join as pjoin

In [10]:
join_path = '/data/akkumar/sae_exps/data/siglip_encoder_23_mean_100_pct'

df_test = pd.read_csv('/data/akkumar/sae_exps/data/test.csv')
df_val = pd.read_csv('/data/akkumar/sae_exps/data/valid.csv')
df_train = pd.read_csv('/data/akkumar/sae_exps/data/train.csv')
df_val['Path'] = df_val['Path'].apply(lambda x: '/'.join(x.split('/')[1:]))
df_train['Path'] = df_train['Path'].apply(lambda x: '/'.join(x.split('/')[1:]))

df_train['split'] = 'train'
df_val['split'] = 'val'
df_test['split'] = 'test'

df = pd.concat([df_train, df_val, df_test], ignore_index=True)
# Need to create a sample_id, split, and input columns, output columns


# Need to get the report, section_findings, and section_impression columns from the reports dataframe
reports = pd.read_csv('/data/akkumar/sae_exps/data/df_master_77_for_release_with_dcm.csv')
# Remove the first 'images' string
reports['path_to_image'] = reports['path_to_image'].apply(lambda x: pjoin(join_path, '_'.join(x.split('/')[1:]).split('.')[0] + ".pt"))
# Rename path_to_image to image_path
reports.rename(columns={'path_to_image': 'image_path'}, inplace=True)


# Create a new column sample_id that is df['path].split('/')[1]
df['sample_id'] = df['Path'].apply(lambda x: '_'.join(x.split('.jpg')[0].split('/')))

# Rename path to image_path
df.rename(columns={'Path': 'image_path'}, inplace=True)

df['image_path'] = df['image_path'].apply(lambda x: pjoin(join_path, '_'.join(x.split('.')[0].split('/')) + ".pt"))

df = df.merge(
    reports[['image_path', 'report', 'section_findings', 'section_impression']],
    on='image_path',
    how='left'
)

# remove space from columns names and put them in lower case with an underline between the words
df.columns = df.columns.str.lower().str.replace(' ', '_')

# Remove NaN and change to -1
df.replace({np.nan: -1}, inplace=True)

In [11]:
df.to_csv('/data/akkumar/sae_exps/data/metadata.csv', index=False)

In [ ]:


# Open this file:
import pickle

with open('/data/akkumar/sae_exps/MechSciDev/tests/chexpert_outputs/linear_probe_results.pkl', 'rb') as f:
    results = pickle.load(f)

print(results['all_metrics'])


{'train': 0.6612152536707037, 'val': 0.9444444444444444, 'test': 0.521181001283697}


: 